# Deepfake XAI Robustness — Kaggle Runner

## Prerequisites (DO THIS FIRST)
1. **Enable Internet**: Settings → Internet → Turn ON
2. **Enable GPU**: Settings → Accelerator → GPU T4 x2 (or P100)
3. **Add Datasets** (click "+ Add Input" on the right panel):
   - Search `ASV spoof 2019` → Add the one with 18 upvotes
   - Search `ASVspoof 2021` → Add `ASVspoof 2021 Dataset`
   - Search `musan` → Add `Starter: musan noise b2c57001-3`
4. **Run cells top to bottom** (Shift+Enter)

In [ ]:
# ============================================================
# CELL 1: Clone Repository
# ============================================================
import os
os.chdir('/kaggle/working')

# Remove any previous clone
!rm -rf deepfake-xai-robustness

# Fresh clone
!git clone https://github.com/shubhikasinha/xai_audio_deepfake.git deepfake-xai-robustness

os.chdir('/kaggle/working/deepfake-xai-robustness')
print(f"\n✓ Working directory: {os.getcwd()}")
print(f"✓ src/data/ exists: {os.path.isdir('src/data')}")
print(f"✓ src/data/utils.py exists: {os.path.isfile('src/data/utils.py')}")

In [ ]:
# ============================================================
# CELL 2: Install Dependencies
# ============================================================
import subprocess, sys

# Install project requirements
!pip install -q -r requirements.txt 2>&1 | tail -3

# Fix NumPy version for PyTorch + Numba compatibility
!pip install -q "numpy>=2.0.0,<2.5.0" --force-reinstall 2>&1 | tail -3

# Install project as editable package
!pip install -q -e . 2>&1 | tail -3

# Verify critical imports work
try:
    import torch
    import numpy as np
    print(f"\n✓ PyTorch: {torch.__version__}")
    print(f"✓ NumPy: {np.__version__}")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"✗ Import error: {e}")

In [ ]:
# ============================================================
# CELL 3: Link Kaggle Datasets
# ============================================================
import os
os.chdir('/kaggle/working/deepfake-xai-robustness')
os.makedirs('data', exist_ok=True)

# --- Discover actual Kaggle input paths ---
def find_and_link(search_root, keywords, link_name):
    """Search /kaggle/input/ for a directory matching keywords and symlink it."""
    if not os.path.isdir(search_root):
        print(f"  ✗ {search_root} does not exist — did you add the dataset?")
        return False
    
    # Walk through input directories to find the right one
    for root, dirs, files in os.walk(search_root):
        depth = root.replace(search_root, '').count(os.sep)
        if depth > 2:  # Don't go too deep
            continue
        dirname = os.path.basename(root)
        # Check if this directory has audio files or expected structure
        if all(kw.lower() in root.lower() for kw in keywords):
            link_path = f'data/{link_name}'
            if os.path.islink(link_path):
                os.unlink(link_path)
            os.symlink(root, link_path)
            print(f"  ✓ {link_name} -> {root}")
            return True
    return False

print("Available Kaggle inputs:")
if os.path.isdir('/kaggle/input'):
    for d in sorted(os.listdir('/kaggle/input')):
        subpath = f'/kaggle/input/{d}'
        print(f"  📁 {d}/")
        # Show first-level subdirectories
        if os.path.isdir(subpath):
            for sd in sorted(os.listdir(subpath))[:5]:
                print(f"      └── {sd}")
else:
    print("  ✗ No /kaggle/input found — datasets not added!")

print("\nLinking datasets...")

# Try standard paths first, then auto-discover
links = {
    'ASVspoof2019_LA': [
        '/kaggle/input/asv-spoof-2019/LA/LA',
        '/kaggle/input/asv-spoof-2019/LA',
        '/kaggle/input/asv-spoof-2019',
    ],
    'ASVspoof2021_DF': [
        '/kaggle/input/asvspoof-2021-dataset/ASVspoof2021_DF_eval/ASVspoof2021_DF_eval',
        '/kaggle/input/asvspoof-2021-dataset/ASVspoof2021_DF_eval',
        '/kaggle/input/asvspoof-2021-dataset',
    ],
    'MUSAN': [
        '/kaggle/input/starter-musan-noise-b2c57001-3/musan',
        '/kaggle/input/starter-musan-noise-b2c57001-3',
        '/kaggle/input/musan',
    ],
}

for link_name, candidates in links.items():
    linked = False
    for path in candidates:
        if os.path.isdir(path):
            link_path = f'data/{link_name}'
            if os.path.islink(link_path):
                os.unlink(link_path)
            os.symlink(path, link_path)
            print(f"  ✓ {link_name} -> {path}")
            linked = True
            break
    if not linked:
        print(f"  ⚠ {link_name}: NOT FOUND — add the dataset via '+ Add Input'")

print("\nFinal data/ layout:")
!ls -la data/

In [ ]:
# ============================================================
# CELL 4: Quick-Test — Verify Pipeline Works
# ============================================================
import os, sys
os.chdir('/kaggle/working/deepfake-xai-robustness')

# Ensure project root is first in sys.path
PROJECT_ROOT = '/kaggle/working/deepfake-xai-robustness'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Force-reload src modules in case of stale imports
import importlib
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

# Now run the quick test
from scripts.run_full_pipeline import main
import sys as _sys
_sys.argv = ['run_full_pipeline', '--quick-test']
main()

---
## ✅ If Cell 4 prints `PASS: ALL COMPONENTS PASSED`, the pipeline is ready.

The cells below are for the actual experiments (run after quick-test passes).

---

In [ ]:
# ============================================================
# CELL 5: Baseline Evaluation — Clean Data (ASVspoof 2021 DF)
# ============================================================
# This cell evaluates pretrained AASIST and WavLM+ECAPA detectors
# on clean (undegraded) ASVspoof 2021 DF data to establish baseline
# EER and min t-DCF metrics.
#
# ⚠ Requires: ASVspoof 2021 dataset linked in Cell 3
# ⚠ Requires: GPU enabled
# ⚠ Runtime: ~15-30 min depending on dataset size

import os, sys
os.chdir('/kaggle/working/deepfake-xai-robustness')
if '/kaggle/working/deepfake-xai-robustness' not in sys.path:
    sys.path.insert(0, '/kaggle/working/deepfake-xai-robustness')

print("Baseline evaluation will be implemented in the next phase.")
print("Quick-test verification must pass first (Cell 4).")